<a href="https://colab.research.google.com/github/RuSlan0000009/RuSlan0000009/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%961_(%D0%A0%D0%B0%D0%B7%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D1%80%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%BE%D0%B9_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D1%8B_%D0%BF%D0%BE_%D0%BF%D0%BE%D0%B4%D0%B1%D0%BE%D1%80%D1%83_%D0%BA%D0%BE%D0%BD%D1%82%D0%B5%D0%BD%D1%82%D0%B0).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Проблема: пользователю сложно выбрать новое аниме для просмотра из огромного количества тайтлов.
Особенно если он знает только одно понравившееся произведение и не понимает, что искать дальше.

Решение: система рекомендаций на основе анализа текстовых описаний, которая автоматически находит похожие произведения.

Цель проекта — разработать систему рекомендаций аниме, которая предсказывает какие тайтлы, на основе описаний и жанра, вероятнее всего понравятся пользователю.

Система помогает пользователям находить новые аниме, если им понравилось какое-то конкретное, даже если они не знают, что искать дальше.

Это задача контентной рекомендательной системы (Content-Based Recommendation).
В проекте не используются оценки пользователей или их историю.
Вместо этого система анализирует содержимое самого аниме — его описание и жанры.

In [ ]:
!pip install opendatasets

In [ ]:
!pip install gradio -q

In [ ]:
!pip install gradio deep-translator -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
import opendatasets as od
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Каталог наборов данных содержит 13 структурированных CSV-файлов, представляющих различные аспекты аниме-экосистемы, собранные с помощью MyAnimeList и Jikan API и охватывающие период с января 1917 года по октябрь 2025 года.

Каждый файл посвящен определенной теме, например: подробности об аниме, персонажах, съемочной группе, пользователях, рейтингах и их взаимосвязях. Вместе они образуют полную реляционную базу данных, содержащую более 130 миллионов записей.

Основные данные аниме: details.csv, stats.csv
Данные о персонаже: characters.csv, character_nicknames.csv, character_anime_works.csv
Данные о персонале и актёрах озвучивания: person_details.csv, person_anime_works.csv, person_voice_works.csv, person_alternate_names.csv
Данные о пользователях и взаимодействии: profiles.csv, ratings.csv, favs.csv, recommendations.csv
Каждый файл может быть связан с другими с помощью согласованных идентификаторов, таких как mal_id, character_mal_id, person_mal_id, и username, что позволяет выполнять мощные реляционные объединения и проводить аналитику по всему аниме-контенту.

Для данного проекта используется набор данных details.csv. Он содержит основные метаданные аниме, включая названия, жанры, студии, оценки, количество серий и информацию о трансляции. Служит основной опорной таблицей для набора данных.

mal_id — Уникальный MyAnimeList идентификатор аниме.
title — Английское или основное название аниме.
title_japanese — Оригинальное японское название.
url — Прямая ссылка на страницу аниме на MAL.
image_url — URL-адрес обложки или постера аниме.
type — Формат аниме (ТВ, фильм, OVA и т. д.).
status — Статус трансляции (в эфире, завершено, планируется).
score — Средняя оценка пользователей по 10-балльной шкале.
scored_by — Количество пользователей, оценивших аниме.
start_date – Дата выхода аниме в эфир.
end_date – Дата окончания показа аниме.
synopsis – Краткое изложение сюжета или описание.
rank – Место в общем рейтинге аниме по количеству баллов.
popularity – Рейтинг популярности на основе интереса пользователей.
members – Общее количество пользователей, добавивших аниме в свои списки.
favorites – Количество пользователей, отметивших аниме как любимое.
genres – Список жанров, связанных с аниме.
studios – Анимационные студии, участвовавшие в производстве.
themes – Тематические теги, описывающие сюжет (например, школа, романтика).
demographics — Категория целевой аудитории (например, Shounen, Seinen).
source — Оригинальный источник (например, манга, лайт-новел, оригинал).
rating — Возрастной рейтинг или рекомендации по содержанию (например, PG-13, R).
episodes — Общее количество эпизодов.
season — Сезон выхода (зима, весна, лето, осень).
year — Год первоначального выхода.
producers — Производственные компании.
explicit_genres — Зрелые или ограниченные жанры (если есть).
licensors — Компании, получившие лицензию на распространение.
streaming — Официальные стриминговые платформы, если таковые имеются.

In [ ]:
od.download("https://www.kaggle.com/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025") #Загружаем датасет

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: rusyarem
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025


100%|██████████| 674M/674M [00:03<00:00, 198MB/s]

In [ ]:
anime_df = pd.read_csv("/content/anime-dataset-jan-1917-to-oct-2025/datasets/details.csv")
anime_df.head(10)

,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
0,59356,-Socket-,-socket-,https://myanimelist.net/anime/59356/-Socket-,https://cdn.myanimelist.net/images/anime/1043/...,Movie,Finished Airing,NaN,NaN,2010-01-01T00:00:00+00:00,...,[],Original,G - All Ages,1.0,NaN,NaN,['Nagoya Zokei University'],[],[],[]
1,56036,......,......,https://myanimelist.net/anime/56036/-,https://cdn.myanimelist.net/images/anime/1057/...,Music,Finished Airing,6.53,503.0,2023-06-11T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1.0,NaN,NaN,[],[],[],[]
2,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745.0,2007-01-18T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,"['Bandai Visual', 'CyberConnect2']",[],[],[]
3,3269,.hack//G.U. Trilogy,.hack//G.U. Trilogy,https://myanimelist.net/anime/3269/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/1566/...,Movie,Finished Airing,7.06,15373.0,2007-12-22T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,['Bandai Visual'],[],"['Funimation', 'Bandai Entertainment']",[]
4,4469,.hack//G.U. Trilogy: Parody Mode,.hack//G.U. Trilogy,https://myanimelist.net/anime/4469/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/10/86...,Special,Finished Airing,6.35,4317.0,2008-03-25T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,['Bandai Visual'],[],[],[]
5,454,.hack//Gift,.hack//GIFT,https://myanimelist.net/anime/454/hack__Gift,https://cdn.myanimelist.net/images/anime/2/230...,OVA,Finished Airing,6.09,10021.0,2003-11-16T00:00:00+00:00,...,[],Original,R+ - Mild Nudity,1.0,NaN,NaN,['CyberConnect2'],[],['Bandai Entertainment'],[]
6,1143,.hack//Intermezzo,.hack//Intermezzo,https://myanimelist.net/anime/1143/hack__Inter...,https://cdn.myanimelist.net/images/anime/1844/...,Special,Finished Airing,6.51,11616.0,2003-03-28T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1.0,NaN,NaN,[],[],['Bandai Entertainment'],['Crunchyroll']
7,299,.hack//Liminality,.hack//LIMINALITY,https://myanimelist.net/anime/299/hack__Limina...,https://cdn.myanimelist.net/images/anime/7/230...,OVA,Finished Airing,6.58,17130.0,2002-06-20T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,4.0,NaN,NaN,"['Bandai Visual', 'Bandai', 'CyberConnect2']",[],['Bandai Entertainment'],[]
8,9332,.hack//Quantum,.hack//Quantum,https://myanimelist.net/anime/9332/hack__Quantum,https://cdn.myanimelist.net/images/anime/1894/...,OVA,Finished Airing,7.11,20412.0,2011-01-28T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,3.0,NaN,NaN,"['Bandai Visual', 'flying DOG', 'Showgate', 'B...",[],['Funimation'],['Crunchyroll']
9,10390,.hack//Quantum: Sore ike! Bokura no Chimuchimu...,.hack//Quantum 映像特典 それいけ！ぼくらのチムチムちゃん！！,https://myanimelist.net/anime/10390/hack__Quan...,https://cdn.myanimelist.net/images/anime/4/290...,Special,Finished Airing,6.29,2810.0,2011-01-28T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,3.0,NaN,NaN,['Bandai Visual'],[],[],[]


In [ ]:
anime_df.info() #Выводим основную информацию о строках и столбцах набора дыннах и проверяем какие строки пропущенны
print('---------------------------------------------------------------------')
print(anime_df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28955 entries, 0 to 28954
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   mal_id           28955 non-null  int64  
 1   title            28955 non-null  object 
 2   title_japanese   28836 non-null  object 
 3   url              28955 non-null  object 
 4   image_url        28955 non-null  object 
 5   type             28888 non-null  object 
 6   status           28955 non-null  object 
 7   score            18882 non-null  float64
 8   scored_by        18882 non-null  float64
 9   start_date       28104 non-null  object 
 10  end_date         11267 non-null  object 
 11  synopsis         23828 non-null  object 
 12  rank             21997 non-null  float64
 13  popularity       28955 non-null  int64  
 14  members          28955 non-null  int64  
 15  favorites        28955 non-null  int64  
 16  genres           28955 non-null  object 
 17  studios     

In [ ]:
print("До удаления Music:", anime_df['type'].value_counts())

# Начинаем предобработку данных
# Оставляем только настоящие аниме (убираем Music)
anime_df = anime_df[anime_df['type'].str.lower() != 'music']

print("После удаления Music:", anime_df['type'].value_counts())

До удаления Music: type
TV            8414
Movie         4915
OVA           4184
ONA           4096
Music         3999
Special       1755
TV Special     767
CM             483
PV             275
Name: count, dtype: int64
После удаления Music: type
TV            8414
Movie         4915
OVA           4184
ONA           4096
Special       1755
TV Special     767
CM             483
PV             275
Name: count, dtype: int64


In [ ]:
# Удаляем ненужные столбцы
anime_df_clear = anime_df.drop(['url', 'type', 'image_url', 'status', 'start_date',
                                'end_date', 'rank', 'popularity', 'members', 'favorites',
                                'studios', 'themes', 'demographics', 'source', 'rating',
                                'episodes', 'season', 'year', 'producers', 'explicit_genres',
                                'licensors', 'streaming', 'mal_id', 'title_japanese',
                                'score', 'scored_by'], axis=1)
# Оставшиесчя столбцы очищаем от пропущенных значений
anime_df_clear = anime_df_clear.dropna(subset=['title', 'synopsis', 'genres'])

# Переименновываем столбцы для удобства
anime_df_clear = anime_df_clear.rename(columns={'title' : 'name', 'synopsis' : 'overview'})

anime_df_clear['type'] = 'anime'

#print(type(anime_df_clear['genres'].iloc[0]))

# Преобразовываем столбец с жанрами к нормальному отображению
anime_df_clear['genres'] = anime_df_clear['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
anime_df_clear['genres'] = anime_df_clear['genres'].apply(lambda x: ', '.join(x))
anime_df_clear = anime_df_clear.reset_index(drop=True)
anime_df_clear.info()
print('-------------------------------------')
print(anime_df_clear.isnull().sum())
print('-------------------------------------')
print(anime_df_clear.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19907 entries, 0 to 19906
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      19907 non-null  object
 1   overview  19907 non-null  object
 2   genres    19907 non-null  object
 3   type      19907 non-null  object
dtypes: object(4)
memory usage: 622.2+ KB
-------------------------------------
name        0
overview    0
genres      0
type        0
dtype: int64
-------------------------------------
                               name  \
0                          -Socket-   
1              .hack//G.U. Returner   
2               .hack//G.U. Trilogy   
3  .hack//G.U. Trilogy: Parody Mode   
4                       .hack//Gift   

                                            overview  \
0  A girl with a cord growing out of her back wan...   
1  The characters from previous .hack//G.U. Games...   
2  Based on the CyberConnect2 HIT GAME, now will ...   
3  A special bonus

In [ ]:
# Подготавливаем данные для обучения модели
anime_df_clear['overview'] = anime_df_clear['overview'].fillna('')
anime_df_clear['genres'] = anime_df_clear['genres'].fillna('')

anime_df_clear['content'] = anime_df_clear['overview'] + ' ' + anime_df_clear['genres']
anime_df_clear['content'] = anime_df_clear['content'].str.lower()


In [ ]:
# Удаляем ненужные символы и нормализует пробелы, чтобы текст был пригоден для анализа TF-IDF.
def clean_text(text):
    text = re.sub(r'[^a-zA-Zа-яА-Я0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

anime_df_clear['content'] = anime_df_clear['content'].apply(clean_text)

In [ ]:
anime_df_clear.info()
anime_df_clear.isnull().sum()
print(anime_df_clear.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19907 entries, 0 to 19906
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      19907 non-null  object
 1   overview  19907 non-null  object
 2   genres    19907 non-null  object
 3   type      19907 non-null  object
 4   content   19907 non-null  object
dtypes: object(5)
memory usage: 777.7+ KB
                               name  \
0                          -Socket-   
1              .hack//G.U. Returner   
2               .hack//G.U. Trilogy   
3  .hack//G.U. Trilogy: Parody Mode   
4                       .hack//Gift   

                                            overview  \
0  A girl with a cord growing out of her back wan...   
1  The characters from previous .hack//G.U. Games...   
2  Based on the CyberConnect2 HIT GAME, now will ...   
3  A special bonus Parody Mode added to the extra...   
4  As an expression of gratitude for the heroes o...   

            

In [ ]:
# Преобразовываем текст в числа для с которыми может работать алгоритм
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(anime_df_clear['content'])

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix) # Cчитаем похожесть каждого аниме с каждым другим аниме на основе их TF-IDF векторов.

In [ ]:
# Создаем удобный индекс по названиям для того чтобы быстро находить его в матрице похожести
indices = pd.Series(anime_df_clear.index, index=anime_df_clear['name']).drop_duplicates()

# Пишем функцию, которая находит выбранное аниме → смотрит, какие аниме ближе всего к нему по TF-IDF + cosine similarity → возвращает топ самых похожих.
def get_recommendations(title, top_n=10):
    if title not in indices:
        return "Такого названия нет в базе"

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    item_indices = [i[0] for i in sim_scores]
    return anime_df_clear[['name', 'overview', 'genres']].iloc[item_indices]

In [ ]:
  get_recommendations("1001 Nights") # Проверяем работу функции

,name,overview,genres
7670,Kachou Fuugetsu,An animated short film by Tatsuo Shimamura.,Avant Garde
12932,Otoko to Onna to Inu,Short animated film by Kuri Youji.,"Avant Garde, Comedy, Romance"
7669,Kachikachi Yama: Umi Yama-hen,"A short Kachikachi Yama film created in 1934, ...",Fantasy
11028,Mikocchaken.,An animated series to promote the film Mikocch...,Comedy
4061,Enyogu,Short film by Yoriko Mizushiri.,Avant Garde
10939,Metamorphose.,Short film by Kiyoshi Nishimoto.,Avant Garde
12483,Okashina Hotel,Short film made by Michio Mihara in 2008.,Avant Garde
14693,Sasurai Zou-san,Short film by Yoriko Mizushiri.,Avant Garde
15555,Shiri Play,Short film by Yoriko Mizushiri.,Avant Garde
16383,Such a Good Place to Die,A short film by Onohana.,Avant Garde


In [ ]:
# Начинаем создавать пользовательский интерфейс
# Делаем умный поиск названия аниме внутри базы, чтобы пользователь мог ввести название с ошибками, частично или в другом регистре, а система всё равно нашла нужный тайтл
from difflib import get_close_matches

titles_original = anime_df_clear['name'].tolist()
titles_lower = [t.lower() for t in titles_original]

def find_title_local(user_input):
    user_input = user_input.lower().strip()

    # точное совпадение
    if user_input in titles_lower:
        return titles_original[titles_lower.index(user_input)]

    # частичное совпадение
    for t in titles_lower:
        if user_input in t:
            return titles_original[titles_lower.index(t)]

    # похожее название
    match = get_close_matches(user_input, titles_lower, n=1, cutoff=0.5)
    if match:
        return titles_original[titles_lower.index(match[0])]

    return None


In [ ]:
# Создаём функцию перевода описания аниме
from deep_translator import GoogleTranslator

translator_en_ru = GoogleTranslator(source='en', target='ru')

def translate_text(text):
    try:
        return translator_en_ru.translate(text)
    except:
        return text  # если перевод не сработал, показываем оригинал

In [ ]:
# Создаём функцию, которая будет связывать модель рекомендаций с интерфейсом

def recommend_anime_ui(user_input, translate_to_russian):
    title = find_title_local(user_input)

    if not title:
        return "Аниме не найдено в загруженной базе.", None

    results = get_recommendations(title, top_n=5)

    text_output = f"Найдено в базе: {title}\n\n"

    for _, row in results.iterrows():
        name = row['name']
        genres = row['genres']
        overview = row['overview'][:400] + "..."

        if translate_to_russian:
            overview = translate_text(overview)

        text_output += f"{name}\nЖанры: {genres}\nОписание: {overview}\n\n"


    return text_output,


In [ ]:
import gradio as gr
# Отображаем пользовательский интерфейс на экране
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("Anime Recommendation System")

    user_input = gr.Textbox(label="Название аниме", placeholder="Например: Naruto")
    translate_checkbox = gr.Checkbox(label="Переводить описания на русский", value=True)

    btn = gr.Button("Найти рекомендации")

    output_text = gr.Textbox(label="Похожие аниме", lines=14)

    btn.click(
        recommend_anime_ui,
        inputs=[user_input, translate_checkbox],
        outputs=[output_text
                 ]
    )

demo.launch(share=True, inline=False)


/tmp/ipython-input-3545465191.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76ac23d4cfd3c75637.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
